# SETUP

In [1]:
# import libraries
from google.colab import files
import pandas as pd
import numpy as np
from sklearn import metrics
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit

Hungarian Data

In [2]:
# load hungarian datafile
##--original
#hungarian_csv_path = "/reprocessed.hungarian.data"
##--modified
hungarian_csv_path = "reprocessed.hungarian.data"

hungarian_data = pd.read_csv(hungarian_csv_path, delimiter = ' ', header = None)
# assign column names
##--original
# hungarian_data.set_axis(['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
#                   'restecg', 'thalach', 'exang', 'oldpeak', 'slope',
#                   'ca', 'thal', 'num'], axis = 1, inplace = True)
##--modified
hungarian_data.columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
                  'restecg', 'thalach', 'exang', 'oldpeak', 'slope',
                  'ca', 'thal', 'num']

Cleveland Data

In [3]:
# load Cleveland datafile
##--original
#cleveland_csv_path = "/processed.cleveland.data"
##--modified
cleveland_csv_path = "processed.cleveland.data"

cleveland_data = pd.read_csv(cleveland_csv_path, header = None)
# assign column names
##--original
# cleveland_data.set_axis(['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
#                   'restecg', 'thalach', 'exang', 'oldpeak', 'slope',
#                   'ca', 'thal', 'num'], axis = 1, inplace = True)
##--modified
cleveland_data.columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
                  'restecg', 'thalach', 'exang', 'oldpeak', 'slope',
                  'ca', 'thal', 'num']

In [4]:
##--modified(unnecessary)
# from google.colab import drive
# drive.mount('/content/drive')

In [5]:
# Define label column
heart_label = 'num'

# SLIGHTLY BETTER MODEL
no missing values & stratified sampling

## DATA PREPROCESSING

### MISSING VALUES

In [6]:
# make copy of original dataframe
cleveland = cleveland_data.copy()

In [7]:
# check for missing values
cleveland.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    float64
 1   sex       303 non-null    float64
 2   cp        303 non-null    float64
 3   trestbps  303 non-null    float64
 4   chol      303 non-null    float64
 5   fbs       303 non-null    float64
 6   restecg   303 non-null    float64
 7   thalach   303 non-null    float64
 8   exang     303 non-null    float64
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    float64
 11  ca        303 non-null    object 
 12  thal      303 non-null    object 
 13  num       303 non-null    int64  
dtypes: float64(11), int64(1), object(2)
memory usage: 33.3+ KB


In [8]:
# replace '?' values with NaN so can impute
##--original
#cleveland['thal'].replace('?', np.NaN, inplace=True)
##--modified
cleveland['thal'].replace('?', np.nan, inplace=True)

cleveland['thal'] = cleveland['thal'].astype(float)
##--original
#cleveland['ca'].replace('?', np.NaN, inplace=True)
##--modified
cleveland['ca'].replace('?', np.nan, inplace=True)

cleveland['ca'] = cleveland['ca'].astype(float)
cleveland.tail() # to verify didn't mess up dataframe IDs

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
298,45.0,1.0,1.0,110.0,264.0,0.0,0.0,132.0,0.0,1.2,2.0,0.0,7.0,1
299,68.0,1.0,4.0,144.0,193.0,1.0,0.0,141.0,0.0,3.4,2.0,2.0,7.0,2
300,57.0,1.0,4.0,130.0,131.0,0.0,0.0,115.0,1.0,1.2,2.0,1.0,7.0,3
301,57.0,0.0,2.0,130.0,236.0,0.0,2.0,174.0,0.0,0.0,2.0,1.0,3.0,1
302,38.0,1.0,3.0,138.0,175.0,0.0,0.0,173.0,0.0,0.0,1.0,NaN,3.0,0


In [9]:
from sklearn.impute import SimpleImputer
# impute with mode as 'thal' and 'ca' (attributes w/ missing values) are discrete
imputeMode = SimpleImputer(strategy="most_frequent") # create mode imputer
imputeMode.fit(cleveland) # fit - learns the data
imputed = imputeMode.transform(cleveland) # transform - imputes with chosen strategy
cleveland = pd.DataFrame(imputed, columns=cleveland.columns, index=cleveland['thal'].index) # back to pandas DataFrame
cleveland.info() # check for missing values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    float64
 1   sex       303 non-null    float64
 2   cp        303 non-null    float64
 3   trestbps  303 non-null    float64
 4   chol      303 non-null    float64
 5   fbs       303 non-null    float64
 6   restecg   303 non-null    float64
 7   thalach   303 non-null    float64
 8   exang     303 non-null    float64
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    float64
 11  ca        303 non-null    float64
 12  thal      303 non-null    float64
 13  num       303 non-null    float64
dtypes: float64(14)
memory usage: 33.3 KB


In [10]:
cleveland.head() # check for anything obviously wonky

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0.0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2.0
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1.0
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0.0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0.0


### BINARY LABEL
(multiclass for "future research")

In [11]:
cleveland['num'][cleveland['num'] > 0] = 1
cleveland.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0.0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1.0
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1.0
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0.0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0.0


### STRATIFIED SAMPLING

(due to clear significance of sex)

In [12]:
cleveland_strat = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in cleveland_strat.split(cleveland, cleveland["num"]):
  cleveland_strat_train = cleveland.loc[train_index]
  cleveland_strat_test = cleveland.loc[test_index]

In [13]:
cleveland_strat_test["num"].value_counts()/len(cleveland_strat_test)

,count
num,
0.0,0.540984
1.0,0.459016


In [14]:
cleveland_strat_train["num"].value_counts()/len(cleveland_strat_train)

,count
num,
0.0,0.541322
1.0,0.458678


Less significant for Cleveland and Statlog datasets, which have less dramatic difference in sex representation in data, but different for other three datasets.

In [15]:
# split stratified data
cleveland_strat_train_X = cleveland_strat_train.drop([heart_label], axis=1)
cleveland_strat_train_y = cleveland_strat_train[heart_label]
cleveland_strat_test_X = cleveland_strat_test.drop([heart_label], axis=1)
cleveland_strat_test_y = cleveland_strat_test[heart_label]
cleveland_strat_train_X.sample(5)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
228,54.0,1.0,4.0,110.0,206.0,0.0,2.0,108.0,1.0,0.0,2.0,1.0,3.0
182,42.0,1.0,1.0,148.0,244.0,0.0,2.0,178.0,0.0,0.8,1.0,2.0,3.0
20,64.0,1.0,1.0,110.0,211.0,0.0,2.0,144.0,1.0,1.8,2.0,0.0,3.0
146,57.0,1.0,4.0,165.0,289.0,1.0,2.0,124.0,0.0,1.0,2.0,3.0,7.0
154,64.0,1.0,4.0,120.0,246.0,0.0,2.0,96.0,1.0,2.2,3.0,1.0,3.0


## Bustin down Pycaret style

In [16]:
pip install pycaret

In [17]:
from pycaret import classification

## Strat Tests...

In [18]:
cleveClass = classification.setup(data = cleveland_strat_train, target = 'num',  normalize = True, transformation = True, remove_multicollinearity = True, multicollinearity_threshold = 0.95)

,Description,Value
0,Session id,4631
1,Target,num
2,Target type,Binary
3,Original data shape,"(242, 14)"
4,Transformed data shape,"(242, 14)"
5,Transformed train set shape,"(169, 14)"
6,Transformed test set shape,"(73, 14)"
7,Numeric features,13
8,Preprocess,True
9,Imputation type,simple


In [19]:
cleveClass = classification.setup(data = cleveland_strat_train, target = 'num')

,Description,Value
0,Session id,7227
1,Target,num
2,Target type,Binary
3,Original data shape,"(242, 14)"
4,Transformed data shape,"(242, 14)"
5,Transformed train set shape,"(169, 14)"
6,Transformed test set shape,"(73, 14)"
7,Numeric features,13
8,Preprocess,True
9,Imputation type,simple


In [20]:
cleveClass.compare_models()

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.8051,0.8676,0.7696,0.8177,0.7832,0.6075,0.6193,0.0690
ridge,Ridge Classifier,0.7985,0.8889,0.7429,0.8121,0.7670,0.5910,0.6023,0.0270
lr,Logistic Regression,0.7926,0.8820,0.7679,0.7883,0.7715,0.5813,0.5893,1.1020
lda,Linear Discriminant Analysis,0.7868,0.8875,0.7179,0.8040,0.7503,0.5666,0.5772,0.0270
ada,Ada Boost Classifier,0.7816,0.8093,0.7589,0.7753,0.7527,0.5599,0.5738,0.1850
nb,Naive Bayes,0.7691,0.8871,0.7179,0.7795,0.7305,0.5334,0.5527,0.0250
rf,Random Forest Classifier,0.7632,0.8511,0.7161,0.7628,0.7292,0.5198,0.5309,0.1810
et,Extra Trees Classifier,0.7515,0.8515,0.7286,0.7428,0.7246,0.4979,0.5116,0.1630
xgboost,Extreme Gradient Boosting,0.7463,0.8225,0.6929,0.7562,0.7115,0.4871,0.4991,0.0880
qda,Quadratic Discriminant Analysis,0.7460,0.8156,0.7321,0.7204,0.7171,0.4883,0.4991,0.0420


Processing:   0%|          | 0/65 [00:00<?, ?it/s]

LGBMClassifier(boosting_type='gbdt', class_weight=None, colsample_bytree=1.0,
               importance_type='split', learning_rate=0.1, max_depth=-1,
               min_child_samples=20, min_child_weight=0.001, min_split_gain=0.0,
               n_estimators=100, n_jobs=-1, num_leaves=31, objective=None,
               random_state=7227, reg_alpha=0.0, reg_lambda=0.0, subsample=1.0,
               subsample_for_bin=200000, subsample_freq=0)

In [21]:
#cleveClass2 = classification.setup(data = cleveland_strat_train, target = 'num',  normalize = True, transformation = True, remove_multicollinearity = True, multicollinearity_threshold = 0.95, train_size = 0.7, test_data =cleveland_strat_test)
cleveClass2 = classification.setup(data = cleveland_strat_train, target = 'num', train_size = 1, test_data =cleveland_strat_test)


,Description,Value
0,Session id,4547
1,Target,num
2,Target type,Binary
3,Original data shape,"(303, 14)"
4,Transformed data shape,"(303, 14)"
5,Transformed train set shape,"(242, 14)"
6,Transformed test set shape,"(61, 14)"
7,Numeric features,13
8,Preprocess,True
9,Imputation type,simple


In [22]:
cleveClass2.compare_models()

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lda,Linear Discriminant Analysis,0.8257,0.8906,0.7659,0.8581,0.8037,0.6479,0.6578,0.0270
ridge,Ridge Classifier,0.8215,0.8913,0.7659,0.8499,0.7998,0.6396,0.6493,0.0390
nb,Naive Bayes,0.8180,0.8907,0.7659,0.8298,0.7939,0.6316,0.6367,0.0440
lightgbm,Light Gradient Boosting Machine,0.8175,0.8581,0.7841,0.8295,0.8004,0.6328,0.6399,0.2680
lr,Logistic Regression,0.8173,0.8857,0.7659,0.8392,0.7952,0.6311,0.6408,0.1300
rf,Random Forest Classifier,0.7970,0.8829,0.7220,0.8381,0.7672,0.5893,0.6028,0.1870
et,Extra Trees Classifier,0.7968,0.8897,0.7386,0.8181,0.7699,0.5891,0.5983,0.1540
xgboost,Extreme Gradient Boosting,0.7927,0.8600,0.7477,0.7987,0.7702,0.5818,0.5851,0.0580
gbc,Gradient Boosting Classifier,0.7885,0.8455,0.7659,0.7871,0.7712,0.5752,0.5814,0.1320
qda,Quadratic Discriminant Analysis,0.7847,0.8670,0.7568,0.7856,0.7662,0.5671,0.5729,0.0280


Processing:   0%|          | 0/65 [00:00<?, ?it/s]

LinearDiscriminantAnalysis(covariance_estimator=None, n_components=None,
                           priors=None, shrinkage=None, solver='svd',
                           store_covariance=False, tol=0.0001)

In [23]:
cleveClass3 = classification.setup(data = cleveland, target = 'num',  normalize = True, transformation = True, remove_multicollinearity = True, multicollinearity_threshold = 0.95)

,Description,Value
0,Session id,2917
1,Target,num
2,Target type,Binary
3,Original data shape,"(303, 14)"
4,Transformed data shape,"(303, 14)"
5,Transformed train set shape,"(212, 14)"
6,Transformed test set shape,"(91, 14)"
7,Numeric features,13
8,Preprocess,True
9,Imputation type,simple


In [24]:
cleveClass3.compare_models()

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
ridge,Ridge Classifier,0.8444,0.9105,0.7933,0.8650,0.8199,0.6835,0.6941,0.0790
lda,Linear Discriminant Analysis,0.8444,0.9105,0.7933,0.8650,0.8199,0.6835,0.6941,0.0790
lr,Logistic Regression,0.8348,0.9124,0.7922,0.8451,0.8102,0.6642,0.6746,0.0820
rf,Random Forest Classifier,0.8310,0.8852,0.7844,0.8465,0.8065,0.6572,0.6677,0.2970
nb,Naive Bayes,0.8301,0.9123,0.8033,0.8373,0.8076,0.6562,0.6732,0.0780
knn,K Neighbors Classifier,0.8212,0.8910,0.7744,0.8268,0.7948,0.6372,0.6437,0.0940
et,Extra Trees Classifier,0.8117,0.8891,0.7944,0.8096,0.7917,0.6207,0.6340,0.2020
qda,Quadratic Discriminant Analysis,0.8067,0.8792,0.7622,0.8256,0.7828,0.6100,0.6233,0.1290
lightgbm,Light Gradient Boosting Machine,0.7931,0.8594,0.7322,0.8133,0.7614,0.5803,0.5925,0.1330
xgboost,Extreme Gradient Boosting,0.7831,0.8563,0.7533,0.7795,0.7584,0.5628,0.5727,0.1100


Processing:   0%|          | 0/65 [00:00<?, ?it/s]

RidgeClassifier(alpha=1.0, class_weight=None, copy_X=True, fit_intercept=True,
                max_iter=None, positive=False, random_state=2917, solver='auto',
                tol=0.0001)

In [25]:
from pycaret.classification import *

ridgeModel = create_model('ridge', fold = 10)


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8182,0.9333,0.9000,0.7500,0.8182,0.6393,0.6500
1,0.8636,0.8750,0.8000,0.8889,0.8421,0.7227,0.7258
2,0.9524,1.0000,1.0000,0.9000,0.9474,0.9041,0.9083
3,0.7619,0.9167,0.5556,0.8333,0.6667,0.4928,0.5173
4,0.8095,0.9074,0.7778,0.7778,0.7778,0.6111,0.6111
5,0.8571,0.9273,0.7000,1.0000,0.8235,0.7097,0.7416
6,0.7143,0.8364,0.7000,0.7000,0.7000,0.4273,0.4273
7,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
8,0.8095,0.8818,0.8000,0.8000,0.8000,0.6182,0.6182


Processing:   0%|          | 0/4 [00:00<?, ?it/s]

In [26]:
tuned_Ridgeclf = tune_model(ridgeModel)


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8182,0.9333,0.9000,0.7500,0.8182,0.6393,0.6500
1,0.8636,0.8750,0.8000,0.8889,0.8421,0.7227,0.7258
2,0.9524,1.0000,1.0000,0.9000,0.9474,0.9041,0.9083
3,0.7619,0.9167,0.5556,0.8333,0.6667,0.4928,0.5173
4,0.8095,0.9074,0.7778,0.7778,0.7778,0.6111,0.6111
5,0.8571,0.9273,0.7000,1.0000,0.8235,0.7097,0.7416
6,0.7143,0.8455,0.7000,0.7000,0.7000,0.4273,0.4273
7,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
8,0.8095,0.8818,0.8000,0.8000,0.8000,0.6182,0.6182


Processing:   0%|          | 0/7 [00:00<?, ?it/s]

Fitting 10 folds for each of 10 candidates, totalling 100 fits


Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


In [27]:
predict_model(tuned_Ridgeclf)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Ridge Classifier,0.8022,0.7993,0.7619,0.8000,0.7805,0.6007,0.6013


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,prediction_label
180,48.0,1.0,4.0,124.0,274.0,0.0,2.0,166.0,0.0,0.5,2.0,0.0,7.0,1.0,1
85,44.0,1.0,3.0,140.0,235.0,0.0,2.0,180.0,0.0,0.0,1.0,0.0,3.0,0.0,0
131,51.0,1.0,3.0,94.0,227.0,0.0,0.0,154.0,1.0,0.0,1.0,1.0,7.0,0.0,1
48,65.0,0.0,3.0,140.0,417.0,1.0,2.0,157.0,0.0,0.8,1.0,1.0,3.0,0.0,0
101,34.0,1.0,1.0,118.0,182.0,0.0,2.0,174.0,0.0,0.0,1.0,0.0,3.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11,56.0,0.0,2.0,140.0,294.0,0.0,2.0,153.0,0.0,1.3,2.0,0.0,3.0,0.0,0
116,58.0,1.0,3.0,140.0,211.0,1.0,2.0,165.0,0.0,0.0,1.0,0.0,3.0,0.0,0
248,52.0,1.0,4.0,125.0,212.0,0.0,0.0,168.0,0.0,1.0,1.0,2.0,7.0,1.0,1
12,56.0,1.0,3.0,130.0,256.0,1.0,2.0,142.0,1.0,0.6,2.0,1.0,6.0,1.0,1


## Hungarian/

In [28]:
hungarian_basic = hungarian_data.copy()
# convert unknowns (-9.0) to NaN
##--original
#hungarian_basic.replace(-9.0, np.NaN, inplace=True)
##--modified
hungarian_basic.replace(-9.0, np.nan, inplace=True)

In [29]:
# make copy for binary labeling
hungarian_basic['num'][hungarian_basic['num'] > 0] = 1
hungarian_basic.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,40.0,1.0,2.0,140.0,289.0,0.0,0.0,172.0,0.0,0.0,NaN,NaN,NaN,0.0
1,49.0,0.0,3.0,160.0,180.0,0.0,0.0,156.0,0.0,1.0,2.0,NaN,NaN,1.0
2,37.0,1.0,2.0,130.0,283.0,0.0,1.0,98.0,0.0,0.0,NaN,NaN,NaN,0.0
3,48.0,0.0,4.0,138.0,214.0,0.0,0.0,108.0,1.0,1.5,2.0,NaN,NaN,1.0
4,54.0,1.0,3.0,150.0,NaN,0.0,0.0,122.0,0.0,0.0,NaN,NaN,NaN,0.0


In [30]:
# try dropping slope, ca, and thal
hungarian_basic.drop(['slope', 'ca', 'thal', ], axis=1, inplace=True) # did not fix error

In [31]:
# split the binary data
from sklearn.model_selection import train_test_split
hungarian_basic_train, hungarian_basic_test = train_test_split(hungarian_basic, test_size=0.2, random_state=42)
hungarian_basic_test_labels = hungarian_basic_test[heart_label]
hungarian_basic_test_nolabel = hungarian_basic_test.drop([heart_label], axis=1)
hungarian_basic_train_labels = hungarian_basic_train[heart_label]
hungarian_basic_train_nolabel = hungarian_basic_train.drop([heart_label], axis=1)
hungarian_basic_train.sample(5)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,num
79,49.0,1.0,4.0,130.0,206.0,0.0,0.0,170.0,0.0,0.0,1.0
135,49.0,1.0,3.0,115.0,265.0,0.0,0.0,175.0,0.0,0.0,1.0
145,39.0,1.0,4.0,110.0,273.0,0.0,0.0,132.0,0.0,0.0,0.0
224,33.0,1.0,3.0,120.0,298.0,0.0,0.0,185.0,0.0,0.0,0.0
7,54.0,1.0,2.0,110.0,208.0,0.0,0.0,142.0,0.0,0.0,0.0


more pre processing...

In [32]:
hungarian = hungarian_data.copy()

# convert unknowns (-9.0) to NaN
##--original
#hungarian.replace(-9.0, np.NaN, inplace=True)
##--modified
hungarian.replace(-9.0, np.nan, inplace=True)

hungarian.drop(['slope', 'ca', 'thal'], axis=1, inplace=True)

In [33]:
hungarian.drop(294,axis=0, inplace=True) # may not be necessary later # something may have gone wrong earlier, requiring it now

In [34]:
from sklearn.impute import SimpleImputer
# impute discrete values using mode ()
imputeMode = SimpleImputer(strategy="most_frequent") # create mode imputer
hungarian_disc = pd.concat([hungarian.pop(x) for x in ['fbs', 'restecg', 'exang']], axis=1) # isolate discrete
imputeMode.fit(hungarian_disc) # fit - learns the data
imputed_disc = imputeMode.transform(hungarian_disc) # transform - imputes with chosen strategy
hungarian_disc_imp = pd.DataFrame(imputed_disc, columns=hungarian_disc.columns, index=hungarian.index) # back to pandas DataFrame
hungarian_disc_imp.info() # check for missing values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294 entries, 0 to 293
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   fbs      294 non-null    float64
 1   restecg  294 non-null    float64
 2   exang    294 non-null    float64
dtypes: float64(3)
memory usage: 7.0 KB


In [35]:
# NEED TO COMBINE cat, cont, and remaining
from sklearn.impute import SimpleImputer
# impute continuous values using mean
imputeMean = SimpleImputer(strategy="mean") # create mode imputer
hungarian_cont = pd.concat([hungarian.pop(x) for x in ['trestbps', 'chol', 'thalach']], axis=1) # isolate continuous
imputeMean.fit(hungarian_cont) # fit - learns the data
imputed_cont = imputeMean.transform(hungarian_cont) # transform - imputes with chosen strategy
hungarian_cont_imp = pd.DataFrame(imputed_cont, columns=hungarian_cont.columns, index=hungarian.index) # back to pandas DataFrame
hungarian_cont_imp.info() # check for missing values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294 entries, 0 to 293
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   trestbps  294 non-null    float64
 1   chol      294 non-null    float64
 2   thalach   294 non-null    float64
dtypes: float64(3)
memory usage: 7.0 KB


In [36]:
hungarian_imp = pd.concat((hungarian_cont_imp, hungarian_disc_imp, hungarian), axis=1)

In [37]:
hungarian_imp['num'][hungarian_imp['num'] > 0] = 1


In [38]:
hungarian_imp_strat = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in hungarian_imp_strat.split(hungarian_imp, hungarian_imp["num"]):
  hungarian_imp_strat_train = hungarian_imp.loc[train_index]
  hungarian_imp_strat_test = hungarian_imp.loc[test_index]

In [39]:
# split stratified data
hungarian_imp_strat_train_X = hungarian_imp_strat_train.drop([heart_label], axis=1)
hungarian_imp_strat_train_y = hungarian_imp_strat_train[heart_label]
hungarian_imp_strat_test_X = hungarian_imp_strat_test.drop([heart_label], axis=1)
hungarian_imp_strat_test_y = hungarian_imp_strat_test[heart_label]
hungarian_imp_strat_train_X.sample(5)

,trestbps,chol,thalach,fbs,restecg,exang,age,sex,cp,oldpeak
201,110.0,249.0,150.0,0.0,0.0,0.0,47.0,1.0,1.0,0.0
247,120.0,237.0,150.0,0.0,0.0,1.0,54.0,1.0,3.0,1.5
9,120.0,284.0,120.0,0.0,0.0,0.0,48.0,0.0,2.0,0.0
48,112.0,340.0,184.0,0.0,0.0,0.0,36.0,1.0,3.0,1.0
92,120.0,210.0,148.0,0.0,0.0,0.0,52.0,0.0,2.0,0.0


In [40]:
from pycaret.classification import *
hungarian_pc_setup = setup(data = hungarian_imp_strat_train, test_data = hungarian_imp_strat_test, target = 'num', session_id=42)

,Description,Value
0,Session id,42
1,Target,num
2,Target type,Binary
3,Original data shape,"(294, 11)"
4,Transformed data shape,"(294, 11)"
5,Transformed train set shape,"(235, 11)"
6,Transformed test set shape,"(59, 11)"
7,Numeric features,10
8,Preprocess,True
9,Imputation type,simple


In [41]:
compare_models()

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
ridge,Ridge Classifier,0.8304,0.9024,0.7208,0.8012,0.7516,0.6246,0.6326,0.0260
lr,Logistic Regression,0.8303,0.8921,0.7208,0.8057,0.7506,0.6240,0.6355,0.1010
lda,Linear Discriminant Analysis,0.8263,0.9016,0.7208,0.7914,0.7470,0.6163,0.6240,0.0400
nb,Naive Bayes,0.8085,0.8986,0.7681,0.7277,0.7392,0.5899,0.5983,0.0270
et,Extra Trees Classifier,0.8043,0.8925,0.6611,0.7841,0.7001,0.5601,0.5755,0.2000
qda,Quadratic Discriminant Analysis,0.8000,0.8742,0.7431,0.7368,0.7256,0.5708,0.5848,0.0250
gbc,Gradient Boosting Classifier,0.7877,0.8826,0.6639,0.7394,0.6776,0.5258,0.5414,0.1890
lightgbm,Light Gradient Boosting Machine,0.7746,0.8636,0.6736,0.7089,0.6684,0.5030,0.5206,0.0570
ada,Ada Boost Classifier,0.7745,0.7775,0.6028,0.7354,0.6525,0.4907,0.5034,0.1100
rf,Random Forest Classifier,0.7705,0.8815,0.6139,0.7180,0.6543,0.4865,0.4949,0.1860


Processing:   0%|          | 0/65 [00:00<?, ?it/s]

RidgeClassifier(alpha=1.0, class_weight=None, copy_X=True, fit_intercept=True,
                max_iter=None, positive=False, random_state=42, solver='auto',
                tol=0.0001)

In [42]:
ridge = create_model('ridge')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8333,0.9778,0.6667,0.8571,0.7500,0.6279,0.6391
1,0.8750,0.9111,0.7778,0.8750,0.8235,0.7273,0.7303
2,0.7917,0.8667,0.6667,0.7500,0.7059,0.5455,0.5477
3,0.7083,0.8000,0.4444,0.6667,0.5333,0.3333,0.3478
4,0.7917,0.9185,0.7778,0.7000,0.7368,0.5652,0.5674
5,0.8696,0.8583,0.6250,1.0000,0.7692,0.6849,0.7217
6,0.8696,0.9417,0.8750,0.7778,0.8235,0.7206,0.7238
7,0.8696,0.9333,0.7500,0.8571,0.8000,0.7039,0.7073
8,0.8696,0.9333,0.8750,0.7778,0.8235,0.7206,0.7238


Processing:   0%|          | 0/4 [00:00<?, ?it/s]

In [43]:
tuned_ridge = tune_model(ridge)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8333,0.9704,0.6667,0.8571,0.7500,0.6279,0.6391
1,0.8750,0.9185,0.7778,0.8750,0.8235,0.7273,0.7303
2,0.7917,0.8370,0.6667,0.7500,0.7059,0.5455,0.5477
3,0.7083,0.7852,0.4444,0.6667,0.5333,0.3333,0.3478
4,0.8333,0.9333,0.7778,0.7778,0.7778,0.6444,0.6444
5,0.8696,0.8583,0.6250,1.0000,0.7692,0.6849,0.7217
6,0.8261,0.9667,0.7500,0.7500,0.7500,0.6167,0.6167
7,0.9130,0.9333,0.7500,1.0000,0.8571,0.7965,0.8135
8,0.9130,0.9167,1.0000,0.8000,0.8889,0.8189,0.8327


Processing:   0%|          | 0/7 [00:00<?, ?it/s]

Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [44]:
predict_model(tuned_ridge)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Ridge Classifier,0.8644,0.8308,0.7143,0.8824,0.7895,0.6911,0.6995


,trestbps,chol,thalach,fbs,restecg,exang,age,sex,cp,oldpeak,num,prediction_label
293,130.0,182.000000,148.0,0.0,0.0,0.0,53.0,1.0,4.0,0.0,0.0,0
246,120.0,171.000000,137.0,0.0,0.0,0.0,54.0,1.0,1.0,2.0,0.0,0
171,120.0,243.000000,160.0,0.0,0.0,0.0,29.0,1.0,2.0,0.0,0.0,0
290,120.0,166.000000,180.0,0.0,0.0,0.0,36.0,1.0,2.0,0.0,0.0,0
287,140.0,250.848709,140.0,0.0,0.0,0.0,59.0,1.0,4.0,0.0,0.0,0
207,120.0,308.000000,180.0,0.0,2.0,0.0,35.0,1.0,2.0,0.0,0.0,0
140,160.0,331.000000,94.0,0.0,0.0,1.0,52.0,1.0,4.0,2.5,1.0,1
257,130.0,394.000000,150.0,0.0,2.0,0.0,55.0,0.0,2.0,0.0,0.0,0
104,118.0,186.000000,124.0,0.0,0.0,0.0,46.0,1.0,4.0,0.0,1.0,0
83,160.0,196.000000,165.0,0.0,0.0,0.0,52.0,1.0,2.0,0.0,0.0,0
